# 01 - Dados, EDA e Pre-processamento

Notebook 1 do projeto. Aqui ficam: carga dos dados, checagens, analise exploratoria, construcao de `X` e `y`, split treino/teste e padronizacao.

## 1) Setup

In [ ]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

SEED = 42
ROOT = Path("..").resolve()
DATA_RAW = ROOT / "data" / "raw"
DATA_PROCESSED = ROOT / "data" / "processed"
MODELS_DIR = ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)

## 2) Coleta e carga

Os arquivos de origem estao em `data/raw`. Para manter o notebook leve, usamos o dataset consolidado em `data/processed/dataset_municipios.csv`.

In [ ]:
raw_files = sorted([p.name for p in DATA_RAW.glob("*") if p.is_file()])
raw_files

In [ ]:
df = pd.read_csv(DATA_PROCESSED / "dataset_municipios.csv")
df.head()

## 3) Qualidade e EDA rapida

In [ ]:
print("shape:", df.shape)
print("duplicados em cod_ibge:", int(df["cod_ibge"].duplicated().sum()))
print("nulos totais:", int(df.isna().sum().sum()))

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
df["alta_violencia"].value_counts().sort_index().plot(kind="bar", ax=ax[0], color=["#4c78a8", "#f58518"])
ax[0].set_title("Distribuicao de y")
ax[0].set_xlabel("alta_violencia")
ax[0].set_ylabel("quantidade")

df["taxa_homicidios"].plot(kind="hist", bins=40, ax=ax[1], color="#54a24b")
ax[1].set_title("Taxa de homicidios")
plt.tight_layout()
plt.show()

In [ ]:
corr_target = (
    df.corr(numeric_only=True)["alta_violencia"]
    .drop("alta_violencia")
    .sort_values(key=lambda s: s.abs(), ascending=False)
)
corr_target.head(10)

## 4) X, y, N e p

In [ ]:
FEATURES = [c for c in df.columns if c not in ["cod_ibge", "taxa_homicidios", "alta_violencia"]]
TARGET = "alta_violencia"

X = df[FEATURES].copy()
y = df[TARGET].astype(int).copy()

N, p = X.shape
print("N:", N)
print("p:", p)
print("classe (pct):", y.value_counts(normalize=True).round(4).to_dict())

## 5) Split e padronizacao

Split estratificado 80/20: preserva classes e mantem teste isolado para avaliacao final. Metodo escolhido: `StandardScaler`, importante para SVM e rede neural.

In [ ]:
X_train_df, X_test_df, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y,
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_df)
X_test = scaler.transform(X_test_df)

print("treino:", X_train.shape, y_train.shape)
print("teste :", X_test.shape, y_test.shape)

## 6) Exportacao dos artefatos

In [ ]:
np.save(DATA_PROCESSED / "X_train.npy", X_train)
np.save(DATA_PROCESSED / "X_test.npy", X_test)
np.save(DATA_PROCESSED / "y_train.npy", y_train.to_numpy())
np.save(DATA_PROCESSED / "y_test.npy", y_test.to_numpy())
(DATA_PROCESSED / "feature_names.txt").write_text("\n".join(FEATURES), encoding="utf-8")
joblib.dump(scaler, MODELS_DIR / "scaler.joblib")

print("Arquivos salvos em data/processed e models/scaler.joblib")